# المحترف في AI — خادم معالجة مؤقت على Google Colab

هذا الخيار للتجربة فقط باستخدام موارد Colab المجانية المتاحة لحسابك. إذا اخترت GPU من Runtime فسيحاول الدفتر تثبيت ONNX Runtime CUDA واستخدام GPU؛ وإلا يعمل على CPU. الجلسة قد تنتهي والرابط العام يتغير عند إعادة التشغيل. لا يوفر 10 خوادم أو 70 مهمة يوميًا ولا تشغيلًا دائمًا. لا تدفع لأي خدمة.

شغّل الخلايا بالترتيب. لا تشارك رابط النفق أو مفتاح الوصول مع أي شخص.

In [ ]:
# 1) تنزيل المشروع وتثبيت متطلبات النظام
!git clone --depth 1 https://github.com/esame4619-coder/Face-AI.git /content/Face-AI
!apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libglib2.0-0 libsm6 libxext6 libxrender1 libgomp1
%cd /content/Face-AI/backend
!pip install -q -r requirements.txt
# Optional open-source EDSR x2 super-resolution model for the image enhancer.
!python -m pip install -q opencv-contrib-python-headless==4.10.0.84
!mkdir -p /content/models
!wget -q https://github.com/Saafke/EDSR_Tensorflow/raw/master/models/EDSR_x2.pb -O /content/models/EDSR_x2.pb
import os, cv2
os.environ['AI_ENHANCE_MODEL_PATH'] = '/content/models/EDSR_x2.pb'
if not hasattr(cv2, 'dnn_superres'):
    raise RuntimeError('OpenCV dnn_superres module is unavailable')
print('EDSR x2 AI image enhancement model installed.')


In [ ]:
# 2) تثبيت FaceFusion 3.9.2 واختيار CUDA عند توفر GPU في جلسة Colab
import subprocess, os
gpu_available = subprocess.run(['bash','-lc','nvidia-smi'], capture_output=True).returncode == 0
install_target = 'cuda@12' if gpu_available else 'default'
print('GPU detected; installing:', install_target) if gpu_available else print('No GPU detected; using CPU')
!git clone --depth 1 --branch 3.9.2 https://github.com/facefusion/facefusion.git /content/facefusion
%cd /content/facefusion
if gpu_available:
    !python install.py cuda@12 --skip-conda
else:
    !python install.py default --skip-conda
!python -m pip install -q opencv-python-headless==5.0.0.93
if gpu_available:
    import onnxruntime as ort
    if 'CUDAExecutionProvider' not in ort.get_available_providers():
        raise RuntimeError('CUDAExecutionProvider was not installed; switch Colab Runtime to GPU or rerun CPU mode.')
    print('ONNX providers:', ort.get_available_providers())
    os.environ['FACEFUSION_EXECUTION_PROVIDERS'] = 'cuda cpu'
else:
    os.environ['FACEFUSION_EXECUTION_PROVIDERS'] = 'cpu'

# Download and verify the detector/swapper models before exposing the temporary API.
from facefusion.program import create_program
from facefusion.args import apply_args
from facefusion import state_manager
from facefusion.processors.modules.face_swapper import core as face_swapper
warmup_args = [
    'headless-run', '--source-paths', '/tmp/mohtarif-source.jpg',
    '--target-path', '/tmp/mohtarif-target.jpg', '--output-path', '/tmp/mohtarif-output.jpg',
    '--processors', 'face_swapper', '--face-detector-model', 'yolo_face',
    '--face-swapper-model', 'inswapper_128'
]
parser = create_program()
parsed = vars(parser.parse_args(warmup_args))
apply_args(parsed, state_manager.init_item)
if not face_swapper.pre_check():
    raise RuntimeError('FaceFusion model download/check failed; do not start the API.')
print('FaceFusion detector and swapper models are ready.')


In [ ]:
# 3) إعداد مفتاح سري وتشغيل API
import os, secrets, getpass, threading, time
os.environ['FACEFUSION_PATH'] = '/content/facefusion/facefusion.py'
os.environ['PYTHON_PATH'] = 'python'
os.environ['API_KEY'] = getpass.getpass('ضع مفتاحًا سريًا قويًا (لا تشاركه): ').strip()
if len(os.environ['API_KEY']) < 24:
    raise ValueError('استخدم مفتاحًا لا يقل عن 24 حرفًا')
os.environ['PORT'] = '8000'
os.chdir('/content/Face-AI/backend')
import uvicorn
config = uvicorn.Config('main:app', host='0.0.0.0', port=8000, log_level='info')
server = uvicorn.Server(config)
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
time.sleep(5)
print('API thread started. Check /health in the next cell.')


In [ ]:
# 4) فحص صحة المحرك وموفّر التنفيذ قبل فتح الرابط العام
import requests, json, onnxruntime as ort
health = requests.get('http://127.0.0.1:8000/health', timeout=30)
print(health.status_code, json.dumps(health.json(), ensure_ascii=False, indent=2))
print('Execution providers:', ort.get_available_providers())
if health.status_code != 200 or not health.json().get('engine_available'):
    raise RuntimeError('FaceFusion غير جاهز؛ لا تستخدم رابط هذا الخادم داخل التطبيق.')
if gpu_available and 'CUDAExecutionProvider' not in ort.get_available_providers():
    raise RuntimeError('GPU متاح لكن CUDAExecutionProvider غير متاح؛ أعد تشغيل التثبيت.')


In [ ]:
# 5) فتح نفق HTTPS مؤقت مجاني عبر Cloudflare Quick Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
import subprocess, re, time
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
public_url = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        time.sleep(1); continue
    print(line.rstrip())
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0); break
if not public_url:
    raise RuntimeError('لم يظهر رابط النفق. افحص رسائل cloudflared وأعد تشغيل الخلية.')
print('\nرابط الخادم المؤقت:', public_url)
print('مفتاح الوصول هو المفتاح الذي أدخلته في الخلية 3؛ احتفظ به لنفسك فقط.')

## ربط التطبيق
في التطبيق > الإعدادات، ضع رابط `trycloudflare.com` الظاهر أعلاه في رابط مدير المعالجة، وضع نفس المفتاح السري الذي أدخلته في الخلية 3، ثم احفظ واختبر الاتصال.

**تنبيهات:** هذا خادم تجريبي واحد وليس مدير مجموعة ولا عدة عمال. عند اختيار GPU يعتمد الأداء على توفر GPU المجاني في حساب Colab؛ لا توجد حصة مضمونة. عند توقف جلسة Colab يتوقف الخادم، ويجب تشغيل الدفتر من جديد واستخدام الرابط الجديد. لا ترفع صورًا أو فيديوهات حساسة إلى نفق عام. لا تترك الخادم مفتوحًا دون مراقبة.